# Workshop: Hand-crafted CNNs for Face Image Classification

Welcome! In this notebook, you will build a **small convolutional neural network (CNN)** from scratch for a binary face-image classification task, and then inspect its predictions with **Grad-CAM**.

## Learning goals

By the end of this notebook, you should be able to:

- inspect and prepare an image dataset for deep learning,
- build a simple CNN in PyTorch,
- train and evaluate a classifier,
- compare train / validation / benchmark performance,
- inspect mistakes made by the model,
- use Grad-CAM to visualize which image regions influenced a prediction.

## Important note about the task

This workshop uses dataset-provided labels (`female` / `male`) as a **simple binary image-classification example**. These labels are a simplification of the dataset and should **not** be interpreted as a reliable way to infer a person's gender identity from appearance.

Please treat this as a teaching example about model training, evaluation, and interpretation.

---
### How to work through the notebook

You will see three kinds of cells:

- **Explanation** cells give you context.
- **Your turn** cells ask you to make a choice or answer a short question.
- **Code** cells run the pipeline.

Try to pause briefly at each question before running the next cell.

## 1. Setup
We start by installing a few packages and importing the libraries we need.

**Checkpoint question:**  
Why do we need different libraries here? Which ones are for:
1. data handling,
2. deep learning,
3. plotting / interactivity,
4. explainability?

In [ ]:
%pip install -q gitpython opencv-python grad-cam

In [ ]:

import json
import random
import shutil
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import git
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython.display import display
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from sklearn import metrics
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
except ImportError:
    IN_COLAB = False

## 2. Get the data

The workshop dataset is stored in a small public repository.

We will use:
- `female/` and `male/` for model development,
- `benchmark/` as a held-out dataset for final evaluation.

The benchmark set is important because it lets us test the model on images it never saw during training or validation.

In [ ]:

if IN_COLAB:
    DATA_PATH = Path('/content/drive/MyDrive/cvlab')
else:
    DATA_PATH = Path.cwd() / 'data'

DATA_PATH.mkdir(exist_ok=True, parents=True)

FACES_PATH = DATA_PATH / 'faces'
FEMALE_PATH = FACES_PATH / 'female'
MALE_PATH = FACES_PATH / 'male'
BENCHMARK_PATH = FACES_PATH / 'benchmark'

if FACES_PATH.is_dir():
    shutil.rmtree(FACES_PATH)

git.Repo.clone_from('https://github.com/susuter/faces_red.git', FACES_PATH)
print("Dataset downloaded to:", FACES_PATH)

### 2.1 Explore the folder structure

Before training anything, it is good practice to inspect:
- the class folders,
- the number of images per class,
- a few example images.

**Your turn:**  
Look at the counts below. Do the classes appear balanced?  
If not, why might that matter during training?

In [ ]:

def count_jpgs_in_directory(dir_name):
    num_jpgs = len(list(Path(dir_name).glob('*.jpg')))
    print(f"{Path(dir_name).name:>10}: {num_jpgs} images for folder {dir_name}")

count_jpgs_in_directory(FEMALE_PATH)
count_jpgs_in_directory(MALE_PATH)
count_jpgs_in_directory(BENCHMARK_PATH / "female")
count_jpgs_in_directory(BENCHMARK_PATH / "male")

In [ ]:

def scroll_face_images(root_folder):
    root_folder = Path(root_folder)
    image_paths, labels = [], []

    for label_dir in sorted(root_folder.iterdir()):
        if not label_dir.is_dir():
            continue
        for fpath in sorted(label_dir.iterdir()):
            if fpath.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                image_paths.append(fpath)
                labels.append(label_dir.name)

    if not image_paths:
        print("No images found.")
        return

    def show(i=0):
        fig, ax = plt.subplots(figsize=(4, 4))
        img = Image.open(image_paths[i]).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{labels[i]} | {image_paths[i].name}")
        ax.axis("off")
        plt.show()

    slider = widgets.IntSlider(value=0, min=0, max=len(image_paths)-1, step=1, description='Image')
    ui = widgets.VBox([slider])
    out = widgets.interactive_output(show, {'i': slider})
    display(ui, out)

scroll_face_images(FACES_PATH)

## 3. Build a balanced training subset

For a workshop, we usually work with a **balanced subset** of manageable size. This keeps training reasonably fast and makes comparisons easier.

### Suggested workflow
1. Pick the same number of images from each class.
2. Keep the number small enough that training finishes in a workshop session.
3. Write down your choice.

**Your turn:**  
Choose how many images per class you want to use.  
Then answer: *Why is this a reasonable choice for this workshop?*

In [ ]:

FEMALE_PATH_SEL = DATA_PATH / 'female_selected'
MALE_PATH_SEL = DATA_PATH / 'male_selected'

for p in [FEMALE_PATH_SEL, MALE_PATH_SEL]:
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

def randomly_select_n_images(in_path: Path, out_path: Path, n: int):
    files = sorted(list(in_path.glob('*.jpg')))
    assert n <= len(files), f"Requested {n} images but only {len(files)} available in {in_path}"
    chosen = random.sample(files, n)
    for src in chosen:
        shutil.copy(src, out_path / src.name)

In [ ]:

# === YOUR CHOICE ===
n_f = 500
n_m = 500

randomly_select_n_images(FEMALE_PATH, FEMALE_PATH_SEL, n_f)
randomly_select_n_images(MALE_PATH, MALE_PATH_SEL, n_m)

print("Selected images:")
count_jpgs_in_directory(FEMALE_PATH_SEL)
count_jpgs_in_directory(MALE_PATH_SEL)

## 4. Pre-process the images

Neural networks expect images with a consistent size and numeric format.

Here we will:
- resize all images to the same height and width,
- convert them to RGB,
- scale pixel values to `[0, 1]`,
- store them in NumPy arrays.

### Label convention
We will use:
- `0` = female
- `1` = male

In a real project, always document your label mapping clearly.

In [ ]:

IMG_SIZE = 96
LABEL_FEMALE = 0
LABEL_MALE = 1

image_list_f = sorted(FEMALE_PATH_SEL.glob('*.jpg'))
image_list_m = sorted(MALE_PATH_SEL.glob('*.jpg'))

def img_preprocessing(image_list, label: int, img_size: int = IMG_SIZE):
    preprocessed_images, labels = [], []

    for image_path in image_list:
        img = cv2.imread(str(image_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
        img = img.astype(np.float32) / 255.0
        preprocessed_images.append(img)
        labels.append(label)

    return preprocessed_images, labels

print("Pre-processing female images...")
preprocessed_image_list_f, labels_f = img_preprocessing(image_list_f, LABEL_FEMALE)

print("Pre-processing male images...")
preprocessed_image_list_m, labels_m = img_preprocessing(image_list_m, LABEL_MALE)

X = np.array(preprocessed_image_list_f + preprocessed_image_list_m, dtype=np.float32)
y = np.array(labels_f + labels_m, dtype=np.int64)

print("X shape:", X.shape)
print("y shape:", y.shape)

### 4.1 Inspect the processed data

**Checkpoint questions:**
1. What does the shape of `X` mean?
2. What does the shape of `y` mean?
3. Why is it useful that all images now have the same shape?

In [ ]:

def show_img(image_set, i, title=None):
    plt.figure(figsize=(3, 3))
    plt.imshow(image_set[i])
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()

def show_label(i):
    label_name = "female" if y[i] == LABEL_FEMALE else "male"
    show_img(X, i, f"Label: {label_name} ({y[i]})")

widgets.interact(show_label, i=widgets.IntSlider(value=0, min=0, max=len(X)-1, step=1));

In [ ]:

class_names = np.array(["female", "male"])
unique, counts = np.unique(y, return_counts=True)
plt.figure(figsize=(4, 3))
plt.bar(class_names[unique], counts)
plt.title("Class counts in selected training data")
plt.ylabel("Number of images")
plt.show()

## 5. Train / validation split

We now split the selected data into:
- a **training set** used for learning the weights,
- a **validation set** used to monitor generalization during training.

A common split is around 80/20 or 85/15.

**Your turn:**  
Choose a validation fraction.  
What happens if the validation set is:
- too small?
- too large?

In [ ]:

X, y = shuffle(X, y, random_state=42)

VAL_FRACTION = 0.15

X_train, X_validation, y_train, y_validation = train_test_split(
    X, y, test_size=VAL_FRACTION, random_state=42, stratify=y
)

print("Train set:", X_train.shape, y_train.shape)
print("Validation set:", X_validation.shape, y_validation.shape)

## 6. Wrap the data in PyTorch datasets

PyTorch works conveniently with `Dataset` and `DataLoader` objects.  
This gives us mini-batches during training.

In [ ]:

class NumpyClassificationDataset(Dataset):
    """
    Simple dataset wrapping (X, y) from numpy.
    - X: float array (N, H, W, 3) in [0,1]
    - y: integer labels (N,)
    """
    def __init__(self, X, y):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.from_numpy(np.transpose(self.X[idx], (2, 0, 1))).float()
        y = torch.tensor(self.y[idx]).long()
        return x, y

train_dataset = NumpyClassificationDataset(X_train, y_train)
val_dataset = NumpyClassificationDataset(X_validation, y_validation)

# tiny subset for overfit check
overfit_train_dataset = NumpyClassificationDataset(X_train[:10], y_train[:10])
overfit_val_dataset = NumpyClassificationDataset(X_train[:10], y_train[:10])

In [ ]:

# Sanity checks
x_batch, y_batch = next(iter(DataLoader(train_dataset, batch_size=8, shuffle=True)))
print("Batch image tensor shape:", x_batch.shape)
print("Batch label tensor shape:", y_batch.shape)
print("Example labels:", y_batch.tolist())

## 7. Define a hand-crafted CNN

Now we build our own CNN instead of using a pretrained architecture.

### Building blocks
Each convolution block will contain:
- a convolution,
- optional batch normalization,
- ReLU activation,
- optional dropout,
- max pooling.

We will then add a small fully connected classifier head.

**Checkpoint question:**  
Why do we usually add max pooling after convolution blocks?

In [ ]:

class ConvModule(nn.Module):
    def __init__(self, input_channels, output_channels, kernel_size=3, padding=1, dropout_ratio=0.0, use_batchnorm=False):
        super().__init__()
        layers = [nn.Conv2d(input_channels, output_channels, kernel_size, padding=padding)]
        if use_batchnorm:
            layers.append(nn.BatchNorm2d(output_channels))
        layers += [
            nn.ReLU(),
            nn.Dropout2d(dropout_ratio),
            nn.MaxPool2d(kernel_size=2),
        ]
        self.conv_module = nn.Sequential(*layers)

    def forward(self, x):
        return self.conv_module(x)

class HandcraftedCNN(nn.Module):
    def __init__(self, num_classes=2, num_channels=32, dropout_ratio=0.0, use_batchnorm=True):
        super().__init__()
        self.model_parameters = {
            "num_classes": num_classes,
            "num_channels": num_channels,
            "dropout_ratio": dropout_ratio,
            "use_batchnorm": use_batchnorm
        }

        self.features = nn.Sequential(
            ConvModule(3, num_channels, dropout_ratio=dropout_ratio, use_batchnorm=use_batchnorm),
            ConvModule(num_channels, num_channels * 2, dropout_ratio=dropout_ratio, use_batchnorm=use_batchnorm),
            ConvModule(num_channels * 2, num_channels * 4, dropout_ratio=dropout_ratio, use_batchnorm=use_batchnorm),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
            n_features = self.features(dummy).flatten(1).shape[1]

        self.classifier = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.ReLU(),
            nn.Dropout(dropout_ratio),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

    def save_weights(self, path):
        torch.save(self.state_dict(), path)

    def load_weights(self, path, map_location=None):
        self.load_state_dict(torch.load(path, map_location=map_location))

model = HandcraftedCNN(num_channels=32, dropout_ratio=0.10, use_batchnorm=True).to(device)
print(model)

### 7.1 Quick model sanity check

Before training, it is smart to check:
- the output shape,
- that the model can process a batch,
- how many trainable parameters it has.

**Your turn:**  
How many output units should this model have for our task?

In [ ]:

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

with torch.no_grad():
    logits = model(x_batch.to(device))
print("Output shape:", logits.shape)

total_params, trainable_params = count_parameters(model)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 8. Training utilities

We now define helper functions for:
- training,
- validation,
- plotting learning curves,
- computing evaluation metrics.

In [ ]:

def evaluate_model(model, dataset, batch_size=32, device=device):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    y_true, y_pred = [], []
    total_loss = 0.0
    loss_fn = nn.CrossEntropyLoss()

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            total_loss += loss.item() * xb.size(0)
            y_true.extend(yb.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    avg_loss = total_loss / len(dataset)

    return {
        "loss": avg_loss,
        "accuracy": metrics.accuracy_score(y_true, y_pred),
        "precision": metrics.precision_score(y_true, y_pred, zero_division=0),
        "recall": metrics.recall_score(y_true, y_pred, zero_division=0),
        "f1": metrics.f1_score(y_true, y_pred, zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }

def train_with_history(model, train_dataset, val_dataset, loss_fn, optim, epochs, batch_size=32, device=device):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        correct = 0
        total = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optim.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optim.step()

            epoch_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == yb).sum().item()
            total += xb.size(0)

        train_loss = epoch_loss / total
        train_acc = correct / total

        val_metrics = evaluate_model(model, val_dataset, batch_size=batch_size, device=device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_metrics["accuracy"])

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | val_loss={val_metrics['loss']:.4f} | "
            f"train_acc={train_acc:.3f} | val_acc={val_metrics['accuracy']:.3f}"
        )

    return history

def plot_training_performance(history):
    epochs = np.arange(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="validation")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], label="train")
    axes[1].plot(epochs, history["val_acc"], label="validation")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## 9. First check: can the model overfit a tiny dataset?

A very useful debugging trick is to train on just a few examples.

If the model cannot overfit 10 samples, then something is probably wrong in:
- the model,
- the labels,
- the loss,
- the training loop.

**Your turn:**  
Before you run the next cell, make a prediction:  
Should the training accuracy on 10 images become very high?

In [ ]:

tiny_model = HandcraftedCNN(num_channels=32, dropout_ratio=0.0, use_batchnorm=True).to(device)
tiny_loss_fn = nn.CrossEntropyLoss()
tiny_optim = Adam(tiny_model.parameters(), lr=1e-3)

tiny_history = train_with_history(
    tiny_model,
    overfit_train_dataset,
    overfit_val_dataset,
    tiny_loss_fn,
    tiny_optim,
    epochs=20,
    batch_size=5,
    device=device
)
plot_training_performance(tiny_history)

## 10. Train the full model

Now we train on the full training set and monitor validation performance.

### Suggested starting values
- learning rate: `1e-3` or `1e-4`
- batch size: `32`
- epochs: `20` to `30`

**Hands-on exercise:**  
Try one configuration first. Then write down:
1. what happened to training loss,
2. what happened to validation loss,
3. whether you observe signs of overfitting.

In [ ]:

# === STARTING CONFIGURATION ===
learning_rate = 1e-3
batch_size = 32
epochs = 20

model = HandcraftedCNN(num_channels=32, dropout_ratio=0.10, use_batchnorm=True).to(device)
loss_fn = nn.CrossEntropyLoss()
optim = Adam(model.parameters(), lr=learning_rate)

history = train_with_history(
    model,
    train_dataset,
    val_dataset,
    loss_fn,
    optim,
    epochs=epochs,
    batch_size=batch_size,
    device=device
)

In [ ]:
plot_training_performance(history)

### Reflection prompt
Look at the curves and answer:

- Is the model still learning at the end?
- Is there a large gap between train and validation accuracy?
- If yes, what might you try next: more dropout, fewer epochs, a different learning rate, or more data?

In [ ]:

val_metrics = evaluate_model(model, val_dataset, batch_size=batch_size, device=device)
print("Validation metrics:")
for k in ["loss", "accuracy", "precision", "recall", "f1"]:
    print(f"{k:>10}: {val_metrics[k]:.4f}")

ConfusionMatrixDisplay.from_predictions(
    val_metrics["y_true"], val_metrics["y_pred"], display_labels=["female", "male"]
)
plt.title("Validation confusion matrix")
plt.show()

## 11. Save your model

Saving the model is useful so that you can:
- reload it later,
- compare different experiments,
- keep the best version for final evaluation.

In [ ]:

MODELS_PATH = DATA_PATH / "models" / "handcrafted"
MODELS_PATH.mkdir(parents=True, exist_ok=True)

model_json = model.model_parameters
with open(MODELS_PATH / "model_faces_handcrafted.json", "w") as json_file:
    json.dump(model_json, json_file, indent=4)

model.save_weights(MODELS_PATH / "model_faces_handcrafted.pth")
print(f"Saved model configuration and weights to {MODELS_PATH}")

## 12. Evaluate on the benchmark set

The benchmark set acts as a **held-out final test**.  
We only use it after model development, not to tune the model repeatedly.

This is important: if we keep checking the benchmark set during model development, it stops being a fair final test.

In [ ]:

image_list_bench_f = sorted((BENCHMARK_PATH / 'female').glob('*.jpg'))
image_list_bench_m = sorted((BENCHMARK_PATH / 'male').glob('*.jpg'))

preprocessed_bench_image_list_f, bench_labels_f = img_preprocessing(image_list_bench_f, LABEL_FEMALE)
preprocessed_bench_image_list_m, bench_labels_m = img_preprocessing(image_list_bench_m, LABEL_MALE)

X_bench = np.array(preprocessed_bench_image_list_f + preprocessed_bench_image_list_m, dtype=np.float32)
y_bench = np.array(bench_labels_f + bench_labels_m, dtype=np.int64)

benchmark_dataset = NumpyClassificationDataset(X_bench, y_bench)
print("Benchmark set:", X_bench.shape, y_bench.shape)

In [ ]:

bench_metrics = evaluate_model(model, benchmark_dataset, batch_size=batch_size, device=device)

print("Benchmark metrics:")
for k in ["loss", "accuracy", "precision", "recall", "f1"]:
    print(f"{k:>10}: {bench_metrics[k]:.4f}")

ConfusionMatrixDisplay.from_predictions(
    bench_metrics["y_true"], bench_metrics["y_pred"], display_labels=["female", "male"]
)
plt.title("Benchmark confusion matrix")
plt.show()

### Reflection prompt
Compare validation and benchmark performance.

- Are the numbers similar?
- If benchmark performance is worse, what might explain that?
- What does that tell you about generalization?

## 13. Look at individual predictions

Metrics are useful, but individual examples often reveal **what kind of mistakes** the model makes.

Below you can browse benchmark examples and inspect:
- the true label,
- the predicted label,
- the class probabilities.

In [ ]:

# get probabilities on benchmark set
model.eval()
with torch.no_grad():
    xb = torch.from_numpy(np.transpose(X_bench, (0, 3, 1, 2))).float().to(device)
    logits = model(xb)
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = np.argmax(probs, axis=1)

def show_example_prediction(X_, y_true, probs_, i):
    pred = int(np.argmax(probs_[i]))
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))

    axes[0].imshow(X_[i])
    axes[0].set_title(f"True: {class_names[y_true[i]]}\nPred: {class_names[pred]}")
    axes[0].axis("off")

    axes[1].bar(class_names, probs_[i])
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Predicted probabilities")

    plt.tight_layout()
    plt.show()

widgets.interact(
    lambda i: show_example_prediction(X_bench, y_bench, probs, i),
    i=widgets.IntSlider(value=0, min=0, max=len(X_bench)-1, step=1)
);

### 13.1 Misclassified examples

Misclassified examples are especially informative because they show where the model struggles.

**Hands-on exercise:**  
Look at a few wrong predictions and describe:
- Are the faces unusual, low quality, or heavily cropped?
- Do some examples look ambiguous even to a human?
- Do you notice a repeated pattern in the mistakes?

In [ ]:

def show_misclassified(X, y_true, probs_):
    misclassified = [i for i in range(len(y_true)) if y_true[i] != int(np.argmax(probs_[i]))]
    print(f"Number of misclassified images: {len(misclassified)}")
    if not misclassified:
        print("No misclassified images found.")
        return

    def show(idx):
        show_example_prediction(X, y_true, probs_, misclassified[idx])

    widgets.interact(
        show,
        idx=widgets.IntSlider(value=0, min=0, max=len(misclassified)-1, step=1, description='Index')
    )

show_misclassified(X_bench, y_bench, probs)

### 13.2 Correctly classified examples

Also inspect a few correct predictions.  
Sometimes the model is very confident for easy examples and much less confident for hard ones.

In [ ]:

def show_correctly_classified(X, y_true, probs_):
    correct = [i for i in range(len(y_true)) if y_true[i] == int(np.argmax(probs_[i]))]
    print(f"Number of correctly classified images: {len(correct)}")
    if not correct:
        print("No correctly classified images found.")
        return

    idx = random.choice(correct)
    print("Random correctly classified example:")
    show_example_prediction(X, y_true, probs_, idx)

show_correctly_classified(X_bench, y_bench, probs)

## 14. Explainable AI with Grad-CAM

Grad-CAM produces a heatmap that highlights image regions that influenced the model's decision.

This can be useful for:
- building intuition,
- spotting suspicious shortcuts,
- checking whether the model attends to meaningful regions.

### Important caution
Grad-CAM is **not** ground truth.  
It gives a rough explanation, not a perfect causal story.

**Checkpoint question:**  
If Grad-CAM highlights the background instead of the face, what could that mean?

In [ ]:

def _last_conv(model):
    for m in reversed(list(model.modules())):
        if isinstance(m, nn.Conv2d):
            return m
    raise RuntimeError("No Conv2d layer found.")

def show_example_prediction_xai(model, X_, y_, probs_, i):
    pred_idx = int(np.argmax(probs_[i]))
    target_layer = _last_conv(model)

    input_tensor = torch.from_numpy(np.transpose(X_[i], (2, 0, 1))).unsqueeze(0).float().to(device)

    cam = GradCAM(model=model, target_layers=[target_layer])
    grayscale_cam = cam(input_tensor=input_tensor)[0]

    rgb_img = X_[i].copy()
    visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(rgb_img)
    axes[0].set_title(f"True: {class_names[y_[i]]}")
    axes[0].axis("off")

    axes[1].imshow(visualization)
    axes[1].set_title(f"Grad-CAM\nPred: {class_names[pred_idx]}")
    axes[1].axis("off")

    axes[2].bar(class_names, probs_[i])
    axes[2].set_ylim(0, 1)
    axes[2].set_title("Predicted probabilities")

    plt.tight_layout()
    plt.show()

widgets.interact(
    lambda i: show_example_prediction_xai(model, X_bench, y_bench, probs, i),
    i=widgets.IntSlider(value=0, min=0, max=len(X_bench)-1, step=1)
);

## 15. Optional extension exercises

Choose one or two of the following and document your observations:

### Exercise A — Learning rate
Train again with a different learning rate, e.g.:
- `1e-4`
- `5e-4`
- `1e-3`

What changes in convergence speed and final validation performance?

### Exercise B — Dropout
Try a different dropout ratio, e.g.:
- `0.0`
- `0.1`
- `0.3`

Does it reduce overfitting?

### Exercise C — Model width
Change `num_channels` from `32` to:
- `16`
- `64`

How does model capacity affect performance and training time?

### Exercise D — Interpretation
Pick one correct and one incorrect benchmark prediction.  
Use Grad-CAM and write 2–3 sentences about what the model seems to focus on.

## 16. Wrap-up

You have now completed a full deep-learning workflow:

- selected and inspected data,
- pre-processed images,
- built a CNN from scratch,
- trained and validated the model,
- evaluated on a held-out benchmark set,
- inspected mistakes,
- visualized model attention with Grad-CAM.

### Final reflection
Write down short answers to these questions:

1. What was the most important sign that your model was learning?
2. What was the clearest sign of overfitting, if any?
3. Which metric do you trust most for this task, and why?
4. What did Grad-CAM add that plain accuracy did not?

---

# Extra: Foundation Model Adaptation with DINOv3

So far you built a CNN from scratch. Now we take a different approach: instead of
training a network end-to-end, we borrow a **foundation model** — a large network
pre-trained on billions of images — and adapt it for our task.

The model is **DINOv3 ViT-S/16**, released by Meta AI in August 2025 (arXiv:2508.10104).
DINOv3 scales self-supervised vision training to 7B parameters; the ViT-S/16 variant
used here is a distilled version that is fast and lightweight while retaining the
backbone's rich features. Training uses purely self-supervised learning (no labels).

### What you will see
1. Load and freeze the DINOv3 backbone.
2. Extract **CLS-token features** (one vector per image) and **patch-token features**
   (one vector per 16×16 pixel patch).
3. Visualise what the backbone "sees" — without any task-specific training:
   - **PCA of patch tokens** → semantic segmentation-like colourings.
   - **UMAP of CLS tokens** → how the feature space clusters by class.
   - **Self-attention maps** → which patches the model attends to.
4. Train a small MLP on the frozen features.
5. Compare accuracy with the handcrafted CNN from earlier.

In [ ]:
# Extra imports for this section
%pip install umap-learn transformers

In [ ]:

try:
    import umap
except ImportError:
    raise ImportError("Please install umap-learn:  pip install umap-learn")

from sklearn.decomposition import PCA
from transformers import AutoModel

## E.1 Load the DINOv3 backbone

We load the distilled **ViT-S/16** variant via Hugging Face Transformers and
immediately freeze all its weights. We will never update them — they stay exactly
as Meta released them.

To download the model, you need a Hugging Face token. If you don't have one,
you can get one by registering at [this link](https://huggingface.co/docs/hub/security-tokens).

Once you have a token, save it in a file called `.access_token_hf` in the project
root directory.

In [ ]:

with open(".access_token_hf") as f:
    HF_TOKEN = f.read().strip()


In [ ]:
HF_MODEL_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"

dino_backbone = AutoModel.from_pretrained(HF_MODEL_ID, token=HF_TOKEN)
dino_backbone.eval().to(device)
for p in dino_backbone.parameters():
    p.requires_grad = False

DINO_DIM      = 384
DINO_PATCH    = 16
DINO_INPUT    = 224
DINO_GRID     = DINO_INPUT // DINO_PATCH                          # 14
NUM_REG       = getattr(dino_backbone.config, 'num_register_tokens', 0)
PATCH_OFFSET  = 1 + NUM_REG   # skip [CLS, reg_1, ..., reg_n] to reach patch tokens

print(f"Backbone loaded — dim: {DINO_DIM}, grid: {DINO_GRID}×{DINO_GRID}, "
      f"register tokens: {NUM_REG}, patch offset: {PATCH_OFFSET}")

## E.2 Extract features

We resize every image to 224×224, apply ImageNet normalisation, and push it
through the frozen backbone. The backbone returns:

- **CLS token** `(N, 384)` — a single summary vector per image (used for classification).
- **Patch tokens** `(N, 196, 384)` — one vector per 16×16 patch (14×14 grid = 196 patches).

We also request attention weights for the visualisation step.

In [ ]:
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

def extract_dino_features(X, batch_size=64):
    """
    X : float32 numpy (N, H, W, 3) in [0, 1]
    Returns
        cls_feats    : (N, DINO_DIM)
        patch_feats  : (N, DINO_GRID*DINO_GRID, DINO_DIM)
    """
    all_cls, all_patches = [], []
    for start in range(0, len(X), batch_size):
        t = torch.from_numpy(np.transpose(X[start:start+batch_size], (0, 3, 1, 2))).float().to(device)
        t = torch.nn.functional.interpolate(t, size=(DINO_INPUT, DINO_INPUT),
                                             mode='bilinear', align_corners=False)
        t = (t - _MEAN) / _STD
        with torch.no_grad():
            out = dino_backbone(pixel_values=t)
        hs = out.last_hidden_state
        all_cls.append(hs[:, 0].cpu().numpy())                              # CLS token
        all_patches.append(hs[:, PATCH_OFFSET:PATCH_OFFSET + DINO_GRID**2]  # skip registers
                           .cpu().numpy())
    return np.concatenate(all_cls), np.concatenate(all_patches)

print("Extracting train features …")
cls_train,   patches_train   = extract_dino_features(X_train)
print("Extracting validation features …")
cls_val,     patches_val     = extract_dino_features(X_validation)
print("Extracting benchmark features …")
cls_bench,   patches_bench   = extract_dino_features(X_bench)

print(f"CLS tokens  — train: {cls_train.shape}, val: {cls_val.shape}, bench: {cls_bench.shape}")
print(f"Patch tokens — train: {patches_train.shape}")

## E.3 Feature visualisation

### E.3.1 PCA of patch tokens

Each image is divided into a 14×14 grid of patches (16×16 pixels each). Each patch
is described by a 384-dimensional vector. We fit PCA on all training patches and
project to 3 dimensions, which we map to R, G, B.

The result shows **semantic structure without any labels** — DINOv3 assigns
similar colours to semantically similar regions (e.g. hair vs background vs skin).

In [ ]:
N_SAMPLE = 200
idx_sample = np.random.choice(len(cls_train), N_SAMPLE, replace=False)
sample_patches = patches_train[idx_sample].reshape(-1, DINO_DIM)

pca3 = PCA(n_components=3)
pca3.fit(sample_patches)

def pca_patch_image(patch_tokens):
    """Turn (DINO_GRID^2, DINO_DIM) patch tokens into an RGB PCA image."""
    proj = pca3.transform(patch_tokens)
    for ch in range(3):
        lo, hi = proj[:, ch].min(), proj[:, ch].max()
        proj[:, ch] = (proj[:, ch] - lo) / (hi - lo + 1e-8)
    return proj.reshape(DINO_GRID, DINO_GRID, 3)

def show_pca_patches(i):
    fig, axes = plt.subplots(1, 3, figsize=(10, 3))

    orig_up = np.array(Image.fromarray((X_train[i] * 255).astype(np.uint8))
                       .resize((DINO_INPUT, DINO_INPUT), Image.BILINEAR)) / 255.0

    axes[0].imshow(orig_up)
    axes[0].set_title("Original image")
    axes[0].axis("off")

    pca_img = pca_patch_image(patches_train[i])
    pca_up = np.array(Image.fromarray((pca_img * 255).astype(np.uint8))
                      .resize((DINO_INPUT, DINO_INPUT), Image.NEAREST)) / 255.0
    axes[1].imshow(pca_up)
    axes[1].set_title(f"PCA patch colours ({DINO_GRID}×{DINO_GRID})")
    axes[1].axis("off")

    axes[2].imshow(orig_up)
    axes[2].imshow(pca_up, alpha=0.55)
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    plt.suptitle(f"True label: {class_names[y_train[i]]}", y=1.02)
    plt.tight_layout()
    plt.show()

widgets.interact(
    show_pca_patches,
    i=widgets.IntSlider(value=0, min=0, max=len(X_train)-1, step=1, description="Image")
);

### E.3.2 UMAP of CLS tokens

We project the 384-dimensional CLS tokens to 2-D with UMAP and colour by class.
A clean separation means the frozen backbone already encodes features that are
useful for our binary classification — before any fine-tuning.

In [ ]:
# Combine train + benchmark for a richer picture
cls_all  = np.concatenate([cls_train, cls_bench])
y_all    = np.concatenate([y_train,   y_bench])
split_all = np.array(["train"] * len(cls_train) + ["bench"] * len(cls_bench))

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
emb = reducer.fit_transform(cls_all)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, color_by, title in zip(
    axes,
    [y_all, (split_all == "bench").astype(int)],
    ["Coloured by class", "Coloured by split (train / bench)"],
):
    labels_plot = [class_names[v] for v in color_by] if title.startswith("Coloured by class")                   else ["bench" if v else "train" for v in color_by]
    unique_labels = sorted(set(labels_plot))
    cmap = plt.cm.get_cmap("tab10", len(unique_labels))
    for k, lbl in enumerate(unique_labels):
        mask = np.array(labels_plot) == lbl
        ax.scatter(emb[mask, 0], emb[mask, 1], s=8, alpha=0.6,
                   color=cmap(k), label=lbl)
    ax.set_title(title)
    ax.legend(markerscale=2)
    ax.axis("off")

plt.suptitle("UMAP of DINOv3 CLS tokens (frozen backbone)", fontsize=13)
plt.tight_layout()
plt.show()

### E.3.3 Self-attention maps

ViT-S/14 has **6 attention heads** in its last transformer block. Each head
attends to different semantic regions. We extract the attention that the CLS token
(the summary token) pays to each image patch and overlay it on the original image.

In [ ]:
# Discover the self-attention module path in this model
for name, mod in dino_backbone.named_modules():
    if hasattr(mod, 'query') and hasattr(mod, 'key') and hasattr(mod, 'value'):
        print(name, '|', type(mod).__name__)

In [ ]:
def _find_last_self_attn(model):
    """Return the last DINOv3ViTAttention module (has q_proj/k_proj/v_proj)."""
    found = None
    for _, mod in model.named_modules():
        if hasattr(mod, 'q_proj') and hasattr(mod, 'k_proj') and hasattr(mod, 'v_proj'):
            found = mod
    if found is None:
        raise RuntimeError("Could not find a self-attention module in the backbone.")
    return found

def get_attention_maps(X_imgs):
    """
    Compute CLS-to-patch self-attention from the last transformer layer.

    In a Vision Transformer, every token attends to every other token via
    scaled dot-product attention: softmax(Q·Kᵀ / √d).  The CLS token is a
    learned summary of the whole image, so its attention row tells us which
    patches it weighted most when forming that summary.

    Concretely, for each attention head we extract row 0 (CLS) of the
    attention matrix and keep only the columns that correspond to image patches
    (skipping the CLS token itself and any register tokens).  Reshaping those
    196 values to a 14×14 grid gives a heatmap over the image.

    Unlike Grad-CAM, this requires no labels or backward pass — the patterns
    emerge purely from the self-supervised pre-training of DINOv3.

    Parameters
    ----------
    X_imgs : np.ndarray, shape (N, H, W, 3), float32 in [0, 1]

    Returns
    -------
    np.ndarray, shape (N, num_heads, DINO_GRID, DINO_GRID)
        Per-head attention heatmaps; values are in [0, 1] (softmax weights).
    """
    t = torch.from_numpy(np.transpose(X_imgs, (0, 3, 1, 2))).float().to(device)
    t = torch.nn.functional.interpolate(t, size=(DINO_INPUT, DINO_INPUT),
                                         mode='bilinear', align_corners=False)
    t = (t - _MEAN) / _STD

    attn_mod = _find_last_self_attn(dino_backbone)
    nh = dino_backbone.config.num_attention_heads

    hidden = {}
    def _pre_hook(module, inp):
        hidden['x'] = inp[0].detach()
    h = attn_mod.register_forward_pre_hook(_pre_hook)

    with torch.no_grad():
        dino_backbone(pixel_values=t)
    h.remove()

    x = hidden['x']                    # (B, N_tokens, D)
    B, N, D = x.shape
    hs = D // nh                        # head size

    def to_heads(y):
        return y.view(B, N, nh, hs).permute(0, 2, 1, 3)

    with torch.no_grad():
        q = to_heads(attn_mod.q_proj(x))
        k = to_heads(attn_mod.k_proj(x))

    # Full attention matrix: (B, nh, N_tokens, N_tokens)
    attn = torch.softmax((q @ k.transpose(-2, -1)) * (hs ** -0.5), dim=-1)
    # Row 0 = CLS token; columns PATCH_OFFSET onward = image patches
    attn_cls = attn[:, :, 0, PATCH_OFFSET:PATCH_OFFSET + DINO_GRID**2]
    return attn_cls.reshape(B, nh, DINO_GRID, DINO_GRID).cpu().numpy()

ATTN_BATCH = 64
attn_train = np.concatenate([
    get_attention_maps(X_train[i:i+ATTN_BATCH])
    for i in range(0, len(X_train), ATTN_BATCH)
])
print(f"Attention maps shape: {attn_train.shape}  (N, num_heads, {DINO_GRID}, {DINO_GRID})")

In [ ]:
def show_attention(i):
    num_heads = attn_train.shape[1]
    fig, axes = plt.subplots(1, num_heads + 1, figsize=(3 * (num_heads + 1), 3))

    orig_up = np.array(Image.fromarray((X_train[i] * 255).astype(np.uint8))
                       .resize((DINO_INPUT, DINO_INPUT), Image.BILINEAR)) / 255.0

    axes[0].imshow(orig_up)
    axes[0].set_title(f"Original\n{class_names[y_train[i]]}")
    axes[0].axis("off")

    for h in range(num_heads):
        am = attn_train[i, h]
        am = (am - am.min()) / (am.max() - am.min() + 1e-8)
        am_up = np.array(Image.fromarray((am * 255).astype(np.uint8))
                         .resize((DINO_INPUT, DINO_INPUT), Image.BILINEAR)) / 255.0
        axes[h + 1].imshow(orig_up)
        axes[h + 1].imshow(am_up, cmap='inferno', alpha=0.55)
        axes[h + 1].set_title(f"Head {h+1}")
        axes[h + 1].axis("off")

    plt.tight_layout()
    plt.show()

widgets.interact(
    show_attention,
    i=widgets.IntSlider(value=0, min=0, max=len(X_train)-1, step=1, description="Image")
);

## E.4 MLP classifier on frozen features

Because we have already extracted the CLS features, training is extremely fast —
we never run images through the backbone again. We train a small 2-layer MLP:

```
Linear(384 → 128) → ReLU → Dropout(0.3) → Linear(128 → 2)
```

We reuse the same `train_with_history` and `evaluate_model` helpers and the same
train / validation / benchmark splits as the handcrafted CNN.

In [ ]:
class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.X = torch.from_numpy(features).float()
        self.y = torch.from_numpy(np.array(labels, dtype=np.int64))
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class DinoMLP(nn.Module):
    def __init__(self, in_dim=DINO_DIM, hidden_dim=128, num_classes=2, dropout=0.3):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )
    def forward(self, x):
        return self.head(x)

dino_train_dataset = FeatureDataset(cls_train, y_train)
dino_val_dataset   = FeatureDataset(cls_val,   y_validation)
dino_bench_dataset = FeatureDataset(cls_bench, y_bench)

In [ ]:
dino_lr     = 1e-3
dino_epochs = 30
dino_bs     = 64

dino_model  = DinoMLP().to(device)
dino_loss   = nn.CrossEntropyLoss()
dino_optim  = torch.optim.Adam(dino_model.parameters(), lr=dino_lr)

dino_history = train_with_history(
    dino_model,
    dino_train_dataset,
    dino_val_dataset,
    dino_loss,
    dino_optim,
    epochs=dino_epochs,
    batch_size=dino_bs,
    device=device,
)
plot_training_performance(dino_history)

In [ ]:
DINO_MODELS_PATH = DATA_PATH / "models" / "dino"
DINO_MODELS_PATH.mkdir(parents=True, exist_ok=True)

torch.save(dino_model.state_dict(), DINO_MODELS_PATH / "dino_mlp_head.pth")
print(f"Saved MLP head to {DINO_MODELS_PATH}")

## E.5 Benchmark evaluation & comparison

Let's see how the DINOv3 + MLP compares to the handcrafted CNN on the same
held-out benchmark set.

In [ ]:
cnn_bench_metrics  = evaluate_model(model, NumpyClassificationDataset(X_bench, y_bench), device=device)
dino_bench_metrics = evaluate_model(dino_model, dino_bench_dataset, device=device)

rows = ["Accuracy", "Precision", "Recall", "F1"]
keys = ["accuracy", "precision", "recall", "f1"]

print(f"{'Metric':<12}  {'Handcrafted CNN':>18}  {'DINOv3 + MLP':>14}")
print("-" * 48)
for row, key in zip(rows, keys):
    print(f"{row:<12}  {cnn_bench_metrics[key]:>18.3f}  {dino_bench_metrics[key]:>14.3f}")

x = np.arange(len(rows))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, [cnn_bench_metrics[k]  for k in keys], width, label="Handcrafted CNN")
ax.bar(x + width/2, [dino_bench_metrics[k] for k in keys], width, label="DINOv3 + MLP")
ax.set_xticks(x)
ax.set_xticklabels(rows)
ax.set_ylim(0, 1.05)
ax.set_title("Benchmark comparison: Handcrafted CNN vs DINOv3 + MLP")
ax.legend()
plt.tight_layout()
plt.show()